# Aesop AbstractGraph embedding and clustering

This experiment embeds semantic node text during AbstractGraph conversion when `EMBED_NODES` is enabled. It turns each AbstractGraph's sparse node matrix into one tale vector by summing its rows, then compares graph clustering with a direct whole-tale text embedding baseline. The clustering cell saves a JSON results manifest with run settings and metrics under `data/processed/`.

Embedding and graph extraction use hosted model APIs and may incur charges. The notebook checkpoints traces and node embeddings under `data/processed/`; do not commit those generated artifacts or credentials. Install with `pip install -e '.[dev,abstractgraph]'` before running.


In [ ]:
from pathlib import Path
import hashlib
import json
import pickle

import numpy as np
from abstractgraph import AbstractGraphTransformer
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

from semantic_graphicalizer import (
    DEFAULT_OPENAI_MODEL,
    DEFAULT_OPENAI_EMBEDDING_MODEL,
    OpenAIEmbeddingClient,
    SemanticGraphicalizer,
    load_aesop_fables,
)
from semantic_graphicalizer.model import as_embedding_client

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
CACHE_DIR = ROOT / "data" / "processed" / "aesop_abstractgraph"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
SMOKE_TEST = False
SMOKE_LIMIT = 3
EMBED_NODES = True
EMBEDDING_MODEL = DEFAULT_OPENAI_EMBEDDING_MODEL
RANDOM_SEED = 17
NBITS = 14

ABSTRACTGRAPH_SETTINGS = {
    "embedding_key": "embedding",  # source key; mapped to base-node `attribute`
    "chunk_key": "chunk_id",
    "parallel_edge_policy": "combine",
    "preserve_direction": True,
    "nbits": NBITS,
}

GRAPH_VECTORIZER_SETTINGS = {
    "return_dense": False,
    "n_jobs": -1,
}


## Load the cached corpus

Titles and source hashes stay in evaluation metadata. `limit=None` loads the complete parsed collection; smoke mode selects the first three tales.


In [ ]:
stories = load_aesop_fables(limit=SMOKE_LIMIT if SMOKE_TEST else None, cache_dir=ROOT / "data" / "raw")
metadata = []
for index, story in enumerate(stories):
    story_hash = hashlib.sha256(story.encode("utf-8")).hexdigest()
    metadata.append({
        "tale_id": f"aesop-{index:04d}-{story_hash[:12]}",
        "source_order": index,
        "title": story.splitlines()[0].strip(),
        "characters": len(story),
        "words": len(story.split()),
        "source_sha256": story_hash,
    })
print(f"Loaded {len(stories)} tales")
metadata[:3]

CONFIG_PATHS = {
    "ontology": ROOT / "configs" / "ontologies" / "aesop.yaml",
    "prompts": ROOT / "configs" / "prompts" / "aesop.yaml",
}
config_hashes = {
    name: hashlib.sha256(path.read_bytes()).hexdigest()
    for name, path in CONFIG_PATHS.items()
}
source_hashes = [row["source_sha256"] for row in metadata]
corpus_sha256 = hashlib.sha256("".join(source_hashes).encode("utf-8")).hexdigest()
trace_config_sha256 = hashlib.sha256(
    json.dumps(
        {"extraction_model": DEFAULT_OPENAI_MODEL, "config_sha256": config_hashes},
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()


## Build or resume semantic traces

Each trace is saved as soon as extraction finishes. Rerunning the cell reuses completed traces; node embeddings are checkpointed in those trace graphs during the next cell.


In [ ]:
graphicalizer = SemanticGraphicalizer(
    ontology=ROOT / "configs" / "ontologies" / "aesop.yaml",
    prompts=ROOT / "configs" / "prompts" / "aesop.yaml",
    embedding_model=EMBEDDING_MODEL,
)
graphicalizer.fit(stories)
traces = []
for story, row in zip(stories, metadata):
    checkpoint = CACHE_DIR / f"{row['tale_id']}-{trace_config_sha256[:8]}.trace.pkl"
    if checkpoint.exists():
        with checkpoint.open("rb") as handle:
            trace = pickle.load(handle)
    else:
        trace = graphicalizer.transform_with_trace([story])[0]
        with checkpoint.open("wb") as handle:
            pickle.dump(trace, handle)
    traces.append(trace)
print(f"Ready: {len(traces)} traces; nodes={sum(t.graph.number_of_nodes() for t in traces)}")


## Convert to AbstractGraph and make one sparse vector per tale

Set `EMBED_NODES=False` to reuse only vectors already attached to trace nodes. With it enabled, conversion computes missing or stale vectors and reuses vectors whose text and embedding configuration match.

`AbstractGraphTransformer` batch-vectorizes the pre-built AbstractGraphs. Its core `vectorize()` path applies each graph's attribute function, adds node-existence and degree features, and pools the node features into one sparse row per tale.


In [ ]:
abstract_graphs = graphicalizer.to_abstract_graphs(
    traces, embed_nodes=EMBED_NODES, **ABSTRACTGRAPH_SETTINGS
)

def keep_prebuilt_abstract_graph(abstract_graph):
    return abstract_graph

graph_matrix = AbstractGraphTransformer(
    nbits=NBITS,
    **GRAPH_VECTORIZER_SETTINGS,
    decomposition_function=keep_prebuilt_abstract_graph,
).fit_transform(abstract_graphs)
for trace, row in zip(traces, metadata):
    checkpoint = CACHE_DIR / f"{row['tale_id']}-{trace_config_sha256[:8]}.trace.pkl"
    with checkpoint.open("wb") as handle:
        pickle.dump(trace, handle)
print(f"AbstractGraphs: {len(abstract_graphs)} | graph vector matrix: {graph_matrix.shape} | nnz={graph_matrix.nnz}")


## Direct whole-tale text baseline

Long tales are split deterministically at whitespace near 6,000 characters. Their chunk embeddings are averaged to produce one text vector per tale.


In [ ]:
def text_chunks(text, max_chars=6000):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        if end < len(text):
            boundary = text.rfind(" ", start + 1, end)
            if boundary > start:
                end = boundary
        chunks.append(text[start:end].strip())
        start = end
    return [chunk for chunk in chunks if chunk]

embedding_client = getattr(graphicalizer, "embedding_client_", None)
if embedding_client is None:
    raw_embedder = graphicalizer.embedder or OpenAIEmbeddingClient(model=EMBEDDING_MODEL)
    embedding_client = as_embedding_client(raw_embedder)
text_parts = [text_chunks(story) for story in stories]
text_cache = CACHE_DIR / "direct_text_vectors.pkl"
if text_cache.exists():
    with text_cache.open("rb") as handle:
        cached = pickle.load(handle)
else:
    cached = None
if (cached and cached.get("source_hashes") == source_hashes
        and cached.get("embedding_model") == EMBEDDING_MODEL
        and cached.get("chunk_chars") == 6000
        and cached.get("pooling") == "mean"):
    text_vectors = cached["vectors"]
else:
    flat_parts = [part for parts in text_parts for part in parts]
    part_vectors = []
    for start in range(0, len(flat_parts), 128):
        part_vectors.extend(np.asarray(v, dtype=float) for v in embedding_client.embed(flat_parts[start:start + 128]))
    text_vectors = []
    offset = 0
    for parts in text_parts:
        current = part_vectors[offset:offset + len(parts)]
        text_vectors.append(np.mean(current, axis=0))
        offset += len(parts)
    with text_cache.open("wb") as handle:
        pickle.dump({"source_hashes": source_hashes, "embedding_model": EMBEDDING_MODEL, "chunk_chars": 6000, "pooling": "mean", "vectors": text_vectors}, handle)
text_matrix = np.vstack(text_vectors)
print(f"Direct text vectors: {text_matrix.shape}")


## Cluster and compare

Scale the sparse graph representation without centering it. Use TruncatedSVD to make a compact graph view for clustering and display; cluster the dense direct-text baseline separately. Metric scores are inspection aids, not proof of semantic quality.


In [ ]:
n_tales = len(stories)
if n_tales < 3:
    raise ValueError("Clustering inspection needs at least three tales")

scaled_graph = StandardScaler(with_mean=False).fit_transform(graph_matrix)
svd_components = max(2, min(50, n_tales - 1, scaled_graph.shape[1] - 1))
graph_features = TruncatedSVD(n_components=svd_components, random_state=RANDOM_SEED).fit_transform(scaled_graph)
text_features = StandardScaler().fit_transform(text_matrix)

cluster_results = {}
metric_rows = []
max_k = min(8, n_tales - 1)
for representation, features in (("AbstractGraph", graph_features), ("Direct text", text_features)):
    for algorithm in ("kmeans", "agglomerative"):
        for k in range(2, max_k + 1):
            if algorithm == "kmeans":
                estimator = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)
            else:
                estimator = AgglomerativeClustering(n_clusters=k)
            labels = estimator.fit_predict(features)
            key = (representation, algorithm, k)
            cluster_results[key] = labels
            metric_rows.append({
                "representation": representation,
                "algorithm": algorithm,
                "k": k,
                "silhouette": silhouette_score(features, labels),
                "calinski_harabasz": calinski_harabasz_score(features, labels),
                "davies_bouldin": davies_bouldin_score(features, labels),
            })
metric_rows[:5]


In [ ]:
from IPython.display import display

try:
    import pandas as pd
    metrics = pd.DataFrame(metric_rows).sort_values(["representation", "silhouette"], ascending=[True, False])
    display(metrics)
except ImportError:
    metrics = metric_rows
    print(metric_rows)

# Compare graph and text assignments for matching algorithm and cluster count.
agreement_rows = []
for k in range(2, max_k + 1):
    graph_labels = cluster_results[("AbstractGraph", "kmeans", k)]
    text_labels = cluster_results[("Direct text", "kmeans", k)]
    agreement = {
        "algorithm": "kmeans",
        "k": k,
        "ari": float(adjusted_rand_score(graph_labels, text_labels)),
        "nmi": float(normalized_mutual_info_score(graph_labels, text_labels)),
    }
    agreement_rows.append(agreement)
    print(f"k={k}: ARI={agreement['ari']:.3f}, NMI={agreement['nmi']:.3f}")


## Save run settings and results

The next cell writes a JSON manifest under `data/processed/` with the corpus and configuration hashes, embedding and AbstractGraph settings, vector dimensions, clustering metrics, and per-tale assignments. Its filename includes the corpus and settings hashes so different runs are retained.


In [ ]:
node_embedding_dimensions = sorted({
    len(data["embedding"])
    for trace in traces
    for _node, data in trace.graph.nodes(data=True)
    if "embedding" in data
})
run_settings = {
    "smoke_test": SMOKE_TEST,
    "smoke_limit": SMOKE_LIMIT if SMOKE_TEST else None,
    "extraction_model": DEFAULT_OPENAI_MODEL,
    "embedding_model": EMBEDDING_MODEL,
    "embed_nodes": EMBED_NODES,
    "abstractgraph": ABSTRACTGRAPH_SETTINGS,
    "graph_vectorizer": GRAPH_VECTORIZER_SETTINGS,
    "text_baseline": {"chunk_chars": 6000, "pooling": "mean"},
    "random_seed": RANDOM_SEED,
}
run_sha256 = hashlib.sha256(
    json.dumps(
        {"settings": run_settings, "config_sha256": config_hashes}, sort_keys=True
    ).encode("utf-8")
).hexdigest()
run_manifest = {
    "schema_version": 1,
    "corpus": {
        "sha256": corpus_sha256,
        "tale_count": len(stories),
        "tales": [
            {
                key: row[key]
                for key in ("tale_id", "source_order", "title", "characters", "words", "source_sha256")
            }
            for row in metadata
        ],
        "config_sha256": config_hashes,
    },
    "settings": run_settings,
    "run_sha256": run_sha256,
    "dimensions": {
        "node_embeddings": node_embedding_dimensions,
        "graph_vectors": graph_matrix.shape[1],
        "text_vectors": int(text_matrix.shape[1]),
        "graph_svd_components": svd_components,
    },
    "metrics": [
        {key: (float(value) if isinstance(value, np.generic) else value)
         for key, value in row.items()}
        for row in metric_rows
    ],
    "graph_text_agreement": agreement_rows,
    "cluster_assignments": {
        f"{representation}|{algorithm}|k={k}": [int(label) for label in labels]
        for (representation, algorithm, k), labels in cluster_results.items()
    },
}
run_mode = "smoke" if SMOKE_TEST else "full"
manifest_path = CACHE_DIR / f"{run_mode}-{corpus_sha256[:12]}-{run_sha256[:8]}-results.json"
manifest_path.write_text(json.dumps(run_manifest, indent=2) + "\n")
print(f"Saved run manifest: {manifest_path}")


In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import pairwise_distances
from IPython.display import display, Markdown

selected_k = min(4, n_tales - 1)
for representation, features, reducer in (
    ("AbstractGraph", graph_features, None),
    ("Direct text", text_features, PCA(n_components=2, random_state=RANDOM_SEED)),
):
    coordinates = reducer.fit_transform(features) if reducer is not None else features[:, :2]
    labels = cluster_results[(representation, "kmeans", selected_k)]
    plt.figure(figsize=(7, 5))
    plt.scatter(coordinates[:, 0], coordinates[:, 1], c=labels, cmap="tab10", s=24)
    plt.title(f"{representation} clusters (k={selected_k})")
    plt.xlabel("Component 1")
    plt.ylabel("Component 2")
    plt.show()

labels = cluster_results[("AbstractGraph", "kmeans", selected_k)]
for label in sorted(set(labels)):
    members = np.flatnonzero(labels == label)
    distances = pairwise_distances(graph_features[members], metric="euclidean")
    medoid_index = members[int(np.argmin(distances.mean(axis=1)))]
    display(Markdown(f"### Cluster {label} ({len(members)} tales) — representative: {metadata[medoid_index]['title']}"))
    print(stories[medoid_index][:400].replace("\n", " "))
    entity_types = Counter(data.get("type") for _node, data in traces[medoid_index].graph.nodes(data=True))
    relation_names = Counter(data.get("relation") for _node, data in traces[medoid_index].graph.nodes(data=True) if data.get("relation"))
    argument_roles = Counter(data.get("role") for _source, _target, data in traces[medoid_index].graph.edges(data=True))
    print("Entity types:", entity_types.most_common(6))
    print("Relations:", relation_names.most_common(6))
    print("Argument roles:", argument_roles.most_common(6))


## Interpretation and limitations

Inspect excerpts, recurring entity types, relations, argument roles, nearest tales, and boundary cases before drawing conclusions. Titles and morals are post-hoc human-readable signals only. Save the chosen settings and metrics with the notebook outputs when recording the experiment.
